# Modality comparisons


Computational scripts save assignment `-1` first, then random assignments `0–99`.
`PAIRING_VIEW` changes only the loaded/displayed results. Full results exclude `-1`
and give each random assignment equal weight after averaging its internal folds.
Mean ± SD are descriptive, not confidence intervals. Missing assignments are an
error; assignment zero and incomplete batches are never substituted for the full result.
Raw signal preparation runs once in a terminal and includes every eligible MEG subject.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = Path.cwd()
sys.path[:0] = [str(ROOT), str(ROOT.parent), str(ROOT.parent/'LB')]
PAIRING_VIEW = 'matched'  # 'matched': -1; 'full': mean ± SD of IDs 0–99.
N_PAIRINGS = 100
PAIRING_SEED = 2026
from vs0_run_pairings import activate, format_command, pairing_context
activate(PAIRING_VIEW, N_PAIRINGS, PAIRING_SEED, display_only=True)
import vs0_load_and_config as shared_data
from vs0_pairing_results import config_view, path_view, show, safe_load, ResultView
ANALYSIS_OUTPUT_ROOT = shared_data.analysis_output_root(ROOT)
DISPLAY_COMPONENTS = 5
SEED = 2026  # Trial/source-control seed, independent of PAIRING_SEED.
BLOCK_SCALING = True
N_COMPONENTS = 10
CONDITION_MODE = 'average'
MAX_GRAM_GIB = 2.
from vs1_coverage_figures import apply_style, MODALITY_COLORS, SETUP_COLORS
apply_style()

MEG_KIND='paired_coverage'
N_COMPONENTS=5
MODELS=['separate_pca','plssvd']
STABILITY_MODEL='plssvd'
COMPONENT=1
SHUFFLE='pairing'  # pairing, trials, both; inner trial variation remains separate.
N_INNER_ITERATIONS=1
N_SPLITS=5
RIDGE_ALPHA=.01
ROI_LEVEL='region'
ATLAS_IMAGE=None
ATLAS_LABELS=None
CACHE_DIR=shared_data.trial_cache_directory(ROOT)
SCALING_MODE='equal_variance' if BLOCK_SCALING else 'none'
RECOMPUTE=False

## Fit and load modality comparisons

In [ ]:
INPUT_DIR=ANALYSIS_OUTPUT_ROOT/'compare_subspace'/MEG_KIND
fit_models=list(dict.fromkeys([*MODELS,'separate_pca']))
command=['python','-u',str(ROOT/'vs3_0_compare_subspace.py'),'--cache-dir',str(CACHE_DIR),
    '--output-dir',str(INPUT_DIR),'--meg-kind',MEG_KIND,'--models',*fit_models,
    '--dimensions',*[str(k) for k in range(1,N_COMPONENTS+1)],'--n-splits',str(N_SPLITS),
    '--block-scaling',SCALING_MODE,'--seed',str(SEED),'--ridge-alpha',str(RIDGE_ALPHA),
    '--max-gram-gib',str(MAX_GRAM_GIB)]
print(format_command(command))
modality=safe_load(path_view,INPUT_DIR,PAIRING_VIEW,N_PAIRINGS)

## Subspace overlap, matched components, within-modality comparisons, clustering and transformation generalization

Every exported metric is displayed, including fixed-iEEG-PCA reference matching formerly computed only in this notebook. Full views show mean ± SD across assignment means. In-sample, train and test partitions remain separate.

In [ ]:
if modality is not None:
    show(modality, max_components=DISPLAY_COMPONENTS, tables=True, arrays=True)

## Controlled component and weight stability

Each assignment is fitted separately. The terminal batch then aligns fits to -1 and publishes a random-assignment summary. Inner trial fits are aligned and averaged within an assignment first. ROI means use assignment SD; ROI cross-correlation SD describes channel-pair heterogeneity and is labeled separately. Pairwise stability entries share assignments.

In [ ]:
WEIGHT_DIR=ANALYSIS_OUTPUT_ROOT/'weight_stability'/MEG_KIND
command=['python','-u',str(ROOT/'vs3_0_weight_stability.py'),'--cache-dir',str(CACHE_DIR),
    '--output-dir',str(WEIGHT_DIR),'--meg-kind',MEG_KIND,'--models',*fit_models,
    '--conditions',SHUFFLE,'--iterations',str(N_INNER_ITERATIONS),'--components',str(N_COMPONENTS),
    '--block-scaling',SCALING_MODE,'--seed',str(SEED),'--roi-level',ROI_LEVEL,
    '--max-gram-gib',str(MAX_GRAM_GIB)]
if ATLAS_IMAGE is not None:command+=['--atlas-image',str(ATLAS_IMAGE)]
if ATLAS_LABELS is not None:command+=['--atlas-labels',str(ATLAS_LABELS)]
print(format_command(command))
from vs3_0_weight_stability import load_weight_summary
summary_root=WEIGHT_DIR/SHUFFLE if PAIRING_VIEW=='matched' else WEIGHT_DIR/'random_assignment_summary'/SHUFFLE
stability=safe_load(load_weight_summary,summary_root,model=STABILITY_MODEL,component=COMPONENT,
                    meg_kind=MEG_KIND,block_scaling=SCALING_MODE,n_components=N_COMPONENTS)

In [ ]:
if stability is not None:
    for name,value in stability.items():
        if isinstance(value,pd.DataFrame):
            print(name);display(value)
    roi=stability['roi_stability']
    for modality,frame in roi.groupby('modality'):
        fig,ax=plt.subplots(figsize=(12,4))
        ax.errorbar(frame.roi,frame.mean_magnitude,yerr=frame.sd_across_iterations if PAIRING_VIEW=='full' else None,fmt='o',capsize=3)
        ax.set(title=f'{modality}: ROI mean |weight|; {PAIRING_VIEW}',ylabel='Mean |weight|');ax.tick_params(axis='x',rotation=90);plt.show()
    for table,metric in [('iteration_correlations','magnitude_r'),('timecourse_correlations','signed_r')]:
        for modality,frame in stability[table].groupby('modality'):
            if frame.empty:continue
            n=stability['config']['iterations'];matrix=np.full((n,n),np.nan)
            for row in frame.itertuples():
                matrix[row.iteration_a,row.iteration_b]=matrix[row.iteration_b,row.iteration_a]=getattr(row,metric)
            fig,ax=plt.subplots();im=ax.imshow(matrix,vmin=-1,vmax=1,cmap='RdBu_r');fig.colorbar(im,ax=ax)
            ax.set(title=f'{modality}: {table}',xlabel='Assignment / inner iteration',ylabel='Assignment / inner iteration');plt.show()

## Covariance-model metrics using the same view

Configure and prepare these models in vs2_0_cov_models first. All native, fixed-reference and component covariance/correlation metrics are reused.

In [ ]:
BASE_CONFIG_FILE=ANALYSIS_OUTPUT_ROOT/'covariance'/'config.json'
covariance=None
if BASE_CONFIG_FILE.exists():
    from vs1_run_coverage import analysis_directory
    covariance_config=json.loads(BASE_CONFIG_FILE.read_text())
    covariance_config.update(meg_kind=MEG_KIND,n_components=N_COMPONENTS,block_scaling=BLOCK_SCALING)
    config_file=BASE_CONFIG_FILE.with_name('compare_modality_config.json')
    config_file.write_text(json.dumps(covariance_config,indent=2))
    print(format_command(['python','-u',str(ROOT/'vs2_run_cov_models.py'),'--config',str(config_file)]))
    covariance=safe_load(config_view,covariance_config,'models',PAIRING_VIEW,N_PAIRINGS)
else:
    print('Set the raw-data configuration in vs2_0_cov_models first.')

In [ ]:
if covariance is not None:
    show(covariance, max_components=DISPLAY_COMPONENTS, tables=True, arrays=True)